In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from pathlib import Path

In [3]:
import dagshub
dagshub.init(repo_owner='AMR-ITH', repo_name='RealEstateInsights', mlflow=True)
import mlflow

# set the tracking server

mlflow.set_tracking_uri("https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow")

# mlflow experiment

mlflow.set_experiment("Exp 4 - Final best model")

Accessing as AMR-ITH

Initialized MLflow to track repo "AMR-ITH/RealEstateInsights"

Repository AMR-ITH/RealEstateInsights initialized!

<Experiment: artifact_location='mlflow-artifacts:/04f402567cd64c1783b5fda833805dc1', creation_time=1746765098531, experiment_id='6', last_update_time=1746765098531, lifecycle_stage='active', name='Exp 4 - Final best model', tags={}>

In [4]:
# pathlib is a module in Python that provides an object-oriented interface 
# for working with file system paths.
current = Path.cwd()
parent = current.parent

# load the data 
df = pd.read_csv(parent /'data-scraped/interim_data.csv')
df.head()

,apartment_name,appartment_loc,zone,bhk_type,construction_status,carpet_area,bulit_area,super_bulit_area,price_value,nearbylocation,facility,luxury_facility_scores
0,nambiar millennia,Sarjapur Road,east,1,Under Construction,NaN,668.0,NaN,0.53,"[('mahatma vidhyalaya', '400 m'), ('eterssrt m...","['Yoga/Meditation Area', ""Children's Play Area...",69
1,provident capella,Samethanahalli,east,1,New Property,NaN,431.0,480.0,0.55,"[('soukya road', '1.4 km'), ('mvj college of e...","[""Children's Play Area"", 'Creche/Day Care', 'J...",70
2,brigade citre budigere cross,Byrathi,east,1,New Property,NaN,619.0,689.0,0.69,"[('one world international school', '3.2kms'),...","['Pet Park', ""Children's Play Area"", 'Landscap...",50
3,sattva east crest bandapura,Budigere Cross,east,1,New Property,NaN,537.0,598.0,0.70,"[('prerana international school', '700 m'), ('...","['Banquet Hall', 'Creche/Day Care', ""Children'...",74
4,sowparnika columns,Soukya Road,east,1,New Property,486.0,619.0,736.0,0.52,"['Whitefield Kadugodi Metro Station', 'Nexus S...","['Lift(s)', 'Swimming Pool', 'Park', 'Fitness ...",44


In [5]:
def categorize_luxury(score):
    if 0 <= score < 50:
        return 'low'
    elif 50 <= score < 150:
        return 'medium'
    else:
        return 'high'
    
df['luxury_category'] = df['luxury_facility_scores'].apply(categorize_luxury)

In [9]:
df.drop(columns=['carpet_area','super_bulit_area','nearbylocation','facility','apartment_name','appartment_loc','luxury_facility_scores'], inplace=True)

In [10]:
temp_df = df.copy()

X = temp_df.drop(columns=['price_value'])
y = temp_df['price_value']

In [11]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [12]:
# do the basic processing input data

num_cols = ['bulit_area']
nomial_cols = ['zone']
ordinal_cols = ['construction_status','bhk_type','luxury_category']


In [13]:
bhk_type_order = ['1','2','3','4','5','6','7','8','9','10']
construction_status_order = ['New Property','Under Construction', 'Relatively New', 'Moderatly Old', 'Old','undefined']
luxury_facility_scores_order = ['low','medium','high']

In [14]:
# build a preprocessor

prepocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
        ("nominal_encode", OneHotEncoder(handle_unknown="ignore",sparse_output=False), nomial_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[construction_status_order,bhk_type_order,luxury_facility_scores_order]), ordinal_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

prepocessor.set_output(transform="pandas")

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(), ['bulit_area']),
                                ('nominal_encode',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['zone']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['New Property',
                                                             'Under '
                                                             'Construction',
                                                             'Relatively New',
                                                             'Moderatly Old',
                                                             'Old',
                                                             'undefined'],
                                                            ['1', '2', '3', '4',
                                                             '5', '6', '7', '8',
                                                             '9', '10'],
                                                            ['low', 'medium',
                                                             'high']]),
                                 ['construction_status', 'bhk_type',
                                  'luxury_category'])],
                  verbose_feature_names_out=False)

In [15]:
# transform the data

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)

X_train_trans

,bulit_area,zone_east,zone_north,zone_south,zone_west,construction_status,bhk_type,luxury_category
2842,0.197885,0.0,1.0,0.0,0.0,0.0,2.0,1.0
903,0.185153,1.0,0.0,0.0,0.0,2.0,2.0,1.0
3262,0.297475,0.0,1.0,0.0,0.0,2.0,2.0,1.0
109,0.114804,1.0,0.0,0.0,0.0,1.0,1.0,1.0
5602,0.092361,0.0,0.0,0.0,1.0,5.0,1.0,0.0
...,...,...,...,...,...,...,...,...
3772,0.224860,0.0,1.0,0.0,0.0,0.0,3.0,1.0
5191,0.140160,0.0,0.0,1.0,0.0,0.0,2.0,1.0
5226,0.175874,0.0,0.0,1.0,0.0,4.0,2.0,1.0
5390,0.118041,0.0,0.0,0.0,1.0,2.0,1.0,1.0


In [16]:
from sklearn.ensemble import RandomForestRegressor
import optuna

from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score

c:\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
# build the best models

best_rf_params ={'n_estimators': 150,
 'max_depth': 29,
 'max_features': 'log2',
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_samples': 0.5052045211887413,
 'bootstrap': True}


best_rf = RandomForestRegressor(**best_rf_params)

In [18]:
# train the model
best_rf.fit(X_train_trans, y_train)



RandomForestRegressor(max_depth=29, max_features='log2',
                      max_samples=0.5052045211887413, n_estimators=150)

In [19]:
# get the train and test predictions

y_train_pred = best_rf.predict(X_train_trans)
y_test_pred = best_rf.predict(X_test_trans)

# calculate the train and test mae

train_mae = mean_absolute_error(y_train,y_train_pred)
test_mae = mean_absolute_error(y_test,y_test_pred)

# calculate the r2 scores

train_r2 = r2_score(y_train,y_train_pred)
test_r2 = r2_score(y_test,y_test_pred)

# calculate cross val scores

cv_scores = cross_val_score(best_rf,
                            X_train_trans,
                            y_train,cv=3,
                            scoring="neg_mean_absolute_error",
                            n_jobs=-1)

In [20]:
print(f"Train MAE: {train_mae}")
print(f"Test MAE: {test_mae}")
print(f"Train R2: {train_r2}")
print(f"Test R2: {test_r2}")
print(f"Cross val scores: {cv_scores}")
print(f"Cross val mean: {np.mean(cv_scores)}")

Train MAE: 0.30616997078124053
Test MAE: 0.46431718616866424
Train R2: 0.925293301338638
Test R2: 0.8343795617849135
Cross val scores: [-0.48424901 -0.52612392 -0.50608067]
Cross val mean: -0.5054845348031408


In [21]:
# log with mlflow

with mlflow.start_run():
    # set tags
    mlflow.set_tag("model","best rf")

    # log parameters
    mlflow.log_params(best_rf.get_params())

    # log metrics
    mlflow.log_metric("train_mae",train_mae)
    mlflow.log_metric("test_mae",test_mae)
    mlflow.log_metric("train_r2",train_r2)
    mlflow.log_metric("test_r2",test_r2)
    mlflow.log_metric("cv_score",-(cv_scores.mean()))

    # log the stacking regressor
    mlflow.sklearn.log_model(best_rf,"model")

2025/05/12 09:40:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run sassy-cub-297 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/6/runs/f5051b2d19c84e1ca34e0e73732a79f8
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/6
